# Modelisation - prediction du rendement (Yield_tons_per_hectare)

Pipeline comparant 7 modeles (regression lineaire, LightGBM, CatBoost, XGBoost,
EBM, FT-Transformer, AutoGluon Tabular) sur un echantillon de
`dataset_unifie.csv`, avec suivi des experiences via MLflow.

Tout est pilote par le dict `CONFIG` de la cellule suivante : taille de
l'echantillon, strategie de gestion des NaN, methode d'encodage des
categorielles, et liste des modeles a entrainer. Changer une valeur et
relancer les cellules suffit a lancer une nouvelle experience (chaque run est
logue dans MLflow, donc les experiences precedentes restent comparables).

Deux etapes de tracking MLflow distinctes :
- **Comparaison CV** (nested) : un run parent `model_comparison`, un run
  enfant par modele avec ses metriques CV.
- **Modele final** (normal) : le modele EBM, choisi pour son interpretabilite
  (voir justification section "Optimisation"), reoptimise avec Optuna puis
  reentraine, logue dans un run MLflow simple, non-nested.

## Schema de la pipeline

```mermaid
flowchart TD
    A[dataset_unifie.csv] --> B[Echantillonnage CONFIG.sample_size]
    B --> C[Split train_val / test]
    C -->|80%| D[train_val]
    C -->|20%, tenu a l'ecart| E[test]
    D --> F[K-Fold CV x CONFIG.cv_folds]
    F --> G[7 modeles: linreg / lightgbm / catboost / xgboost / ebm / ft_transformer / autogluon]
    G --> H[Metriques CV: rmse, mae, r2, business_cost]
    H --> I[MLflow nested: parent model_comparison + 1 run par modele]
    I --> J[Classement CV a titre informatif]
    J --> K[Optuna: recherche d'hyperparametres sur EBM uniquement, interpretabilite]
    K --> L[Refit sur train_val avec les meilleurs hyperparametres]
    L --> M[Evaluation finale sur test]
    M --> N[MLflow run normal, non-nested]
```

In [ ]:
import os

os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")  # mlflow>=3 desactive le file store par defaut

import mlflow
import mlflow.sklearn
import numpy as np
import optuna
import pandas as pd

from catboost import CatBoostRegressor
from interpret.glassbox import ExplainableBoostingRegressor
from lightgbm import LGBMRegressor
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import make_scorer, mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import KFold, cross_val_score, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from xgboost import XGBRegressor

optuna.logging.set_verbosity(optuna.logging.WARNING)  # les logs par trial noient le notebook

## Configuration

Seul point a modifier pour changer d'experience.

In [ ]:
CONFIG = {
    "csv_path": "processed_data/dataset_unifie.csv",
    "target": "Yield_tons_per_hectare",
    "sample_size": 100000,       # None pour utiliser tout le dataset
    "test_size": 0.2,             # fraction du total, tenue a l'ecart jusqu'a la verification finale
    "cv_folds": 5,                # k-fold CV sur le reste (80%) pour comparer/selectionner les modeles
    "random_state": 42,
    "nan_strategy": "mean",       # "median" "mean" "constant"
    "encoding": "onehot",         # "onehot" "ordinal"
    "cost_over": 2.0,             # cout d'1 tonne/ha de sur-estimation (sur-engagement, penalites)
    "cost_under": 1.0,            # cout d'1 tonne/ha de sous-estimation (ventes manquees)
    "models": ["linreg", "lightgbm", "catboost", "xgboost", "ebm", "ft_transformer", "autogluon"],
    "ft_transformer_max_epochs": 20,
    "ft_transformer_batch_size": 1024,
    "autogluon_time_limit": 120,   # secondes par fit() - borne le temps total de la CV
    "autogluon_preset": "medium_quality",
    "optuna_n_trials": 30,
    "mlflow_experiment": "yield_prediction",
    "mlflow_tracking_uri": "file:./mlruns",
}

## Chargement et echantillonnage

In [8]:
df = pd.read_csv(CONFIG["csv_path"])

if CONFIG["sample_size"] and CONFIG["sample_size"] < len(df):
    df = df.sample(n=CONFIG["sample_size"], random_state=CONFIG["random_state"]).reset_index(drop=True)

print(df.shape)
df.head()

(100000, 11)


,Region,Soil_Type,Crop,Yield_tons_per_hectare,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,Pesticides_tonnes_avg_proxy
0,East,Clay,Maize,4.018176,442.605149,29.860830,True,False,Cloudy,132,13735.498111
1,East,Peaty,Barley,6.574645,825.053474,33.408699,True,False,Cloudy,77,NaN
2,West,Loam,Maize,2.726768,493.472585,37.182444,False,False,Rainy,71,13735.498111
3,North,Clay,Barley,4.523507,449.798887,35.127805,False,True,Sunny,117,NaN
4,West,Peaty,Maize,6.552316,648.721114,34.415111,True,True,Sunny,147,13735.498111


In [9]:
X = df.drop(columns=[CONFIG["target"]])
y = df[CONFIG["target"]]

NUMERIC_FEATURES = X.select_dtypes(include="number").columns.tolist()
CATEGORICAL_FEATURES = X.select_dtypes(include=["object", "string", "bool"]).columns.tolist()

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=CONFIG["test_size"], random_state=CONFIG["random_state"]
)

print("numeriques:", NUMERIC_FEATURES)
print("categorielles:", CATEGORICAL_FEATURES)
print("train_val:", X_train_val.shape, "test:", X_test.shape)

numeriques: ['Rainfall_mm', 'Temperature_Celsius', 'Days_to_Harvest', 'Pesticides_tonnes_avg_proxy']
categorielles: ['Region', 'Soil_Type', 'Crop', 'Fertilizer_Used', 'Irrigation_Used', 'Weather_Condition']
train_val: (80000, 10) test: (20000, 10)


## Preprocessing configurable

`build_preprocessor` construit le `ColumnTransformer` a partir de
`CONFIG["nan_strategy"]` et `CONFIG["encoding"]`. Meme pipeline pour les 5
modeles de `CLASSIC_MODEL_REGISTRY` afin de les comparer sur une base
equitable (FT-Transformer et AutoGluon font leur propre preprocessing, voir
plus bas).

In [10]:
def build_preprocessor(nan_strategy: str, encoding: str) -> ColumnTransformer:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(
            strategy="constant" if nan_strategy == "constant" else nan_strategy,
            fill_value=0,
        )),
        ("scaler", StandardScaler()),
    ])

    if encoding == "onehot":
        encoder = OneHotEncoder(handle_unknown="ignore")
    elif encoding == "ordinal":
        encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    else:
        raise ValueError(f"Encodage inconnu: {encoding}")

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", encoder),
    ])

    return ColumnTransformer([
        ("num", numeric_pipeline, NUMERIC_FEATURES),
        ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
    ])

## Wrappers sklearn pour FT-Transformer et AutoGluon

Ces deux modeles ont besoin des features brutes (categorielles non
encodees) pour faire leur propre preprocessing interne (embeddings pour
FT-Transformer, feature engineering automatique pour AutoGluon) - ils ne
passent donc pas par `build_preprocessor`. Les wrappers exposent
l'interface sklearn standard (`fit`/`predict`, `get_params`/`set_params`
herites de `BaseEstimator`) pour rester utilisables par `cross_validate` et
la recherche Optuna comme n'importe quel autre modele.

In [ ]:
class FTTransformerRegressor(BaseEstimator, RegressorMixin):
    """Wrapper sklearn autour de pytorch-tabular (FT-Transformer)."""

    def __init__(
        self,
        numeric_features,
        categorical_features,
        target_name="target",
        max_epochs=20,
        batch_size=1024,
        learning_rate=1e-3,
        random_state=42,
    ):
        self.numeric_features = numeric_features
        self.categorical_features = categorical_features
        self.target_name = target_name
        self.max_epochs = max_epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.random_state = random_state

    def _build_model(self):
        from pytorch_tabular import TabularModel
        from pytorch_tabular.config import DataConfig, OptimizerConfig, TrainerConfig
        from pytorch_tabular.models import FTTransformerConfig

        data_config = DataConfig(
            target=[self.target_name],
            continuous_cols=list(self.numeric_features),
            categorical_cols=list(self.categorical_features),
        )
        trainer_config = TrainerConfig(
            max_epochs=self.max_epochs,
            batch_size=self.batch_size,
            early_stopping="valid_loss",
            early_stopping_patience=3,
            checkpoints=None,  # pas de dossier saved_models/, on garde le modele en memoire
            progress_bar="none",
            seed=self.random_state,
        )
        model_config = FTTransformerConfig(task="regression", learning_rate=self.learning_rate)
        model = TabularModel(
            data_config=data_config,
            model_config=model_config,
            optimizer_config=OptimizerConfig(),
            trainer_config=trainer_config,
        )
        # sans experiment_config, model.logger vaut None - mais pl.Trainer(logger=None) retombe
        # quand meme sur son TensorBoardLogger par defaut (dossier lightning_logs/). Il faut
        # explicitement False. trainer_kwargs={"logger": False} ne marche pas non plus: pytorch_tabular
        # passe deja logger=self.logger positionnellement, d'ou un conflit de kwarg avec trainer_kwargs.
        model.logger = False
        return model

    def _prepare(self, X):
        # bypasse build_preprocessor (embeddings categorielles => colonnes brutes)
        # mais les NaN du dataset doivent quand meme etre geres, sinon la loss diverge en NaN.
        df = X.copy()
        df[self.numeric_features] = df[self.numeric_features].fillna(self.numeric_medians_)
        df[self.categorical_features] = df[self.categorical_features].fillna("missing").astype(str)
        return df

    def fit(self, X, y):
        self.numeric_medians_ = X[self.numeric_features].median()
        df = self._prepare(X)
        df[self.target_name] = np.asarray(y)
        self.model_ = self._build_model()
        self.model_.fit(train=df)
        return self

    def predict(self, X):
        preds = self.model_.predict(self._prepare(X))
        return preds[f"{self.target_name}_prediction"].to_numpy()


class AutoGluonTabularRegressor(BaseEstimator, RegressorMixin):
    """Wrapper sklearn autour d'AutoGluon TabularPredictor."""

    def __init__(self, target_name="target", time_limit=120, presets="medium_quality", random_state=42):
        self.target_name = target_name
        self.time_limit = time_limit
        self.presets = presets
        self.random_state = random_state

    def fit(self, X, y):
        import tempfile

        from autogluon.tabular import TabularPredictor

        df = X.copy()
        df[self.target_name] = np.asarray(y)
        self.predictor_ = TabularPredictor(
            label=self.target_name,
            problem_type="regression",
            eval_metric="root_mean_squared_error",
            verbosity=0,
            path=tempfile.mkdtemp(prefix="autogluon_"),  # hors du repo, jamais dans AutogluonModels/
        ).fit(train_data=df, time_limit=self.time_limit, presets=self.presets)
        return self

    def predict(self, X):
        return self.predictor_.predict(X).to_numpy()

## Registre de modeles

Deux registres : `CLASSIC_MODEL_REGISTRY` (modeles sklearn-compatibles,
passent par le `ColumnTransformer` commun) et `RAW_MODEL_REGISTRY`
(FT-Transformer, AutoGluon - preprocessing interne). `build_full_estimator`
construit l'estimateur complet (pipeline ou wrapper brut) a partir d'un nom
de modele - utilise a la fois pour la comparaison CV et pour la recherche
Optuna. Ajouter/retirer un modele classique = ajouter/retirer une entree
dans `CLASSIC_MODEL_REGISTRY` et dans `CONFIG["models"]`.

In [ ]:
CLASSIC_MODEL_REGISTRY = {
    "linreg": LinearRegression(),
    "lightgbm": LGBMRegressor(random_state=CONFIG["random_state"], verbosity=-1),
    "catboost": CatBoostRegressor(random_state=CONFIG["random_state"], verbose=False, allow_writing_files=False),
    "xgboost": XGBRegressor(random_state=CONFIG["random_state"], verbosity=0),
    "ebm": ExplainableBoostingRegressor(random_state=CONFIG["random_state"]),
}

RAW_MODEL_REGISTRY = {
    "ft_transformer": FTTransformerRegressor(
        numeric_features=NUMERIC_FEATURES,
        categorical_features=CATEGORICAL_FEATURES,
        target_name=CONFIG["target"],
        max_epochs=CONFIG["ft_transformer_max_epochs"],
        batch_size=CONFIG["ft_transformer_batch_size"],
        random_state=CONFIG["random_state"],
    ),
    "autogluon": AutoGluonTabularRegressor(
        target_name=CONFIG["target"],
        time_limit=CONFIG["autogluon_time_limit"],
        presets=CONFIG["autogluon_preset"],
        random_state=CONFIG["random_state"],
    ),
}


def build_full_estimator(model_name: str):
    if model_name in RAW_MODEL_REGISTRY:
        return clone(RAW_MODEL_REGISTRY[model_name])
    return Pipeline([
        ("preprocessor", build_preprocessor(CONFIG["nan_strategy"], CONFIG["encoding"])),
        ("model", clone(CLASSIC_MODEL_REGISTRY[model_name])),
    ])

## Entrainement + evaluation (cross-validation) + suivi MLflow (nested)

Chaque modele (7 au total, y compris `linreg` pour reference) est evalue par
k-fold CV (`CONFIG["cv_folds"]` folds) sur `train_val` (80%) - c'est la
moyenne CV (rmse, mae, r2, cout metier) qui sert a comparer/selectionner les
modeles. Le modele est ensuite reentraine sur tout `train_val` et evalue une
seule fois sur `test` (20%, tenu a l'ecart) pour la verification finale - ce
score ne doit pas guider les choix.

Tracking MLflow **nested** pour cette etape : un run parent
`model_comparison`, un run enfant par modele. Le modele final (apres
Optuna, plus bas) sera logue separement dans un run MLflow normal,
non-nested.

In [ ]:
mlflow.set_tracking_uri(CONFIG["mlflow_tracking_uri"])
mlflow.set_experiment(CONFIG["mlflow_experiment"])


def business_cost(y_true, y_pred, cost_over=CONFIG["cost_over"], cost_under=CONFIG["cost_under"]):
    # sur-estimer (sur-engagement/penalites) coute cost_over/cost_under fois plus cher que sous-estimer (ventes manquees)
    error = np.asarray(y_pred) - np.asarray(y_true)
    return np.mean(np.where(error > 0, cost_over * error, cost_under * -error))


def eval_split(estimator, X_split, y_split):
    y_pred = estimator.predict(X_split)
    return {
        "rmse": root_mean_squared_error(y_split, y_pred),
        "mae": mean_absolute_error(y_split, y_pred),
        "r2": r2_score(y_split, y_pred),
        "business_cost": business_cost(y_split, y_pred),
    }


SCORING = {
    "rmse": make_scorer(root_mean_squared_error, greater_is_better=False),
    "mae": make_scorer(mean_absolute_error, greater_is_better=False),
    "r2": make_scorer(r2_score),
    "business_cost": make_scorer(business_cost, greater_is_better=False),
}
cv = KFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=CONFIG["random_state"])

results = []

with mlflow.start_run(run_name="model_comparison"):
    for model_name in CONFIG["models"]:
        estimator = build_full_estimator(model_name)

        with mlflow.start_run(run_name=model_name, nested=True):
            mlflow.log_params({
                "model": model_name,
                "nan_strategy": CONFIG["nan_strategy"],
                "encoding": CONFIG["encoding"],
                "sample_size": len(df),
                "test_size": CONFIG["test_size"],
                "cv_folds": CONFIG["cv_folds"],
                "cost_over": CONFIG["cost_over"],
                "cost_under": CONFIG["cost_under"],
            })

            cv_scores = cross_validate(estimator, X_train_val, y_train_val, cv=cv, scoring=SCORING)
            cv_metrics = {}
            for metric in SCORING:
                values = cv_scores[f"test_{metric}"]
                if metric != "r2":
                    values = -values  # make_scorer(greater_is_better=False) renvoie l'oppose du score
                cv_metrics[f"cv_{metric}_mean"] = values.mean()
                cv_metrics[f"cv_{metric}_std"] = values.std()

            estimator.fit(X_train_val, y_train_val)
            test_metrics = eval_split(estimator, X_test, y_test)

            mlflow.log_metrics(cv_metrics)
            mlflow.log_metrics({f"test_{k}": v for k, v in test_metrics.items()})
            mlflow.sklearn.log_model(estimator, name="model", serialization_format="cloudpickle")

            results.append({
                "model": model_name,
                **cv_metrics,
                **{f"test_{k}": v for k, v in test_metrics.items()},
            })
            print(
                f"{model_name}: cv_rmse={cv_metrics['cv_rmse_mean']:.4f} "
                f"cv_business_cost={cv_metrics['cv_business_cost_mean']:.4f} "
                f"test_rmse={test_metrics['rmse']:.4f}"
            )

pd.DataFrame(results).sort_values("cv_business_cost_mean")

## Comparaison des runs

Recupere tous les runs loggues dans l'experience MLflow (y compris ceux de
sessions/configs precedentes) pour comparer encodages, strategies de NaN et
modeles entre eux. Classement sur `cv_business_cost_mean` (la metrique
metier, issue de la cross-validation) ; `test_rmse`/`test_business_cost` ne
sont affiches que pour verification du modele finalement retenu.

In [13]:
runs = mlflow.search_runs(
    experiment_names=[CONFIG["mlflow_experiment"]],
    order_by=["metrics.cv_business_cost_mean ASC"],
)

runs[[
    "run_id", "params.model", "params.encoding", "params.nan_strategy",
    "metrics.cv_rmse_mean", "metrics.cv_business_cost_mean", "metrics.cv_business_cost_std",
    "metrics.test_rmse", "metrics.test_business_cost", "metrics.test_r2",
]]

,run_id,params.model,params.encoding,params.nan_strategy,metrics.cv_rmse_mean,metrics.cv_business_cost_mean,metrics.cv_business_cost_std,metrics.test_rmse,metrics.test_business_cost,metrics.test_r2
0,338a4396b1d740758cd0c031fa60494e,linreg,onehot,median,0.501478,0.601168,0.005247,0.497939,0.599552,0.912502
1,eab86ef47fd440b1a85dcd7851ab0873,linreg,onehot,mean,0.501478,0.601168,0.005247,0.497939,0.599552,0.912502
2,431a595625f64bb2bfa4661a39629a51,lightgbm,onehot,median,0.503939,0.603985,0.005291,0.499800,0.601994,0.911847
3,519eb7652f334cd2ae9480e466a62f23,lightgbm,onehot,mean,0.503900,0.604006,0.005366,0.499563,0.601713,0.911930
4,b2822df49d7445a99b7d2a7aadc516b9,catboost,onehot,mean,0.505532,0.606077,0.005745,0.500751,0.603154,0.911511
5,13d9a607ca4e4ab8b6ad12b955d0da89,catboost,onehot,median,0.505631,0.606327,0.005904,0.500726,0.603142,0.911520
6,77b9073796ca472887c8c71081fa777c,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN


## Classement des modeles (hors regression lineaire)

`linreg` sert de reference mais n'est jamais candidat au titre de "meilleur
modele". Classement sur `cv_business_cost_mean` (la metrique metier issue de
la CV), a titre indicatif : l'optimisation Optuna (section suivante) cible
directement EBM, independamment de ce classement (justification ci-dessous).

In [ ]:
comparison = pd.DataFrame(results).sort_values("cv_business_cost_mean")
best_cv_model_name = comparison.loc[comparison["model"] != "linreg", "model"].iloc[0]

print("Meilleur modele CV (hors linreg, indicatif):", best_cv_model_name)
comparison

## Optimisation des hyperparametres (Optuna)

**Pourquoi seulement EBM ?** Le classement CV ci-dessus montre des scores tres
proches entre LightGBM, CatBoost, XGBoost et EBM (`cv_business_cost_mean` a
moins de 1% d'ecart) : aucun ne se distingue nettement en performance pure.
Dans ce contexte, on privilegie EBM, seul modele du comparatif (avec `linreg`,
trop simple) a etre nativement interpretable "glass-box" - ses fonctions de
forme par variable peuvent etre inspectees et expliquees directement, ce qui
compte pour un cas d'usage agricole ou les predictions de rendement doivent
pouvoir etre justifiees aupres d'experts metier. Optimiser un modele boite
noire (LightGBM/CatBoost/XGBoost/FT-Transformer/AutoGluon) pour un gain de
performance marginal n'apporterait pas cette garantie. Inutile donc de
definir un espace de recherche pour les 6 autres modeles.

`OPTUNA_SEARCH_SPACES` associe a EBM une fonction `trial -> estimateur`
(meme pipeline que `build_full_estimator`, avec des hyperparametres a la
place des valeurs par defaut). L'objectif est evalue par la meme CV que la
comparaison, sur `train_val`, en minimisant `business_cost` moyen - pas de
tracking MLflow par trial (seul le modele final est logue, cf. section
suivante).

In [ ]:
MODEL_TO_OPTIMIZE = "ebm"

OPTUNA_SEARCH_SPACES = {
    "ebm": lambda trial: Pipeline([
        ("preprocessor", build_preprocessor(CONFIG["nan_strategy"], CONFIG["encoding"])),
        ("model", ExplainableBoostingRegressor(
            random_state=CONFIG["random_state"],
            max_bins=trial.suggest_int("max_bins", 128, 512),
            interactions=trial.suggest_int("interactions", 0, 20),
            learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        )),
    ]),
}


def optuna_objective(trial):
    estimator = OPTUNA_SEARCH_SPACES[MODEL_TO_OPTIMIZE](trial)
    scores = cross_val_score(estimator, X_train_val, y_train_val, cv=cv, scoring=SCORING["business_cost"])
    return -scores.mean()


study = optuna.create_study(direction="minimize")
study.optimize(optuna_objective, n_trials=CONFIG["optuna_n_trials"])

print("Meilleurs hyperparametres:", study.best_params)
print("Meilleur cv_business_cost:", study.best_value)

## Modele final : reentrainement + tracking MLflow (run normal, non-nested)

Reconstruction de l'estimateur avec les meilleurs hyperparametres Optuna
(via `optuna.trial.FixedTrial` pour rejouer les memes fonctions de
`OPTUNA_SEARCH_SPACES` sans dupliquer le code), reentrainement sur tout
`train_val`, evaluation finale sur `test` (toujours tenu a l'ecart). Logue
dans **un seul run MLflow**, sans nesting.

In [ ]:
best_estimator = OPTUNA_SEARCH_SPACES[MODEL_TO_OPTIMIZE](optuna.trial.FixedTrial(study.best_params))
best_estimator.fit(X_train_val, y_train_val)
final_test_metrics = eval_split(best_estimator, X_test, y_test)

with mlflow.start_run(run_name=f"final_{MODEL_TO_OPTIMIZE}"):
    mlflow.log_params({"model": MODEL_TO_OPTIMIZE, **study.best_params})
    mlflow.log_metric("optuna_best_cv_business_cost", study.best_value)
    mlflow.log_metrics({f"test_{k}": v for k, v in final_test_metrics.items()})
    mlflow.sklearn.log_model(best_estimator, name="model", serialization_format="cloudpickle")

print(
    f"Modele final ({MODEL_TO_OPTIMIZE}): "
    f"test_rmse={final_test_metrics['rmse']:.4f} "
    f"test_business_cost={final_test_metrics['business_cost']:.4f}"
)

## Interface MLflow

Lance le serveur `mlflow ui` en arriere-plan (non-bloquant) pour explorer les runs dans le navigateur.


In [4]:
import subprocess
import sys

if "mlflow_ui_proc" not in globals() or mlflow_ui_proc.poll() is not None:
    env = os.environ | {"MLFLOW_ALLOW_FILE_STORE": "true"}
    mlflow_ui_proc = subprocess.Popen(
        [sys.executable, "-m", "mlflow", "ui", "--backend-store-uri", CONFIG["mlflow_tracking_uri"]],
        env=env,
    )

print("MLflow UI: http://127.0.0.1:5000")

MLflow UI: http://127.0.0.1:5000


## Explication locale du modele final (SHAP waterfall)

Le classement CV et l'optimisation Optuna ne regardent que des moyennes
agregees. Un waterfall SHAP montre, sur un exemple individuel du jeu de
test, comment chaque variable a pousse la prediction au-dessus ou en
dessous de la valeur moyenne `E[f(X)]`. On explique le modele EBM sur
l'espace transforme par `preprocessor` (`shap.Explainer` sur `model.predict`,
avec les noms de colonnes de `get_feature_names_out()`) plutot que sur les
colonnes brutes : plus simple a mettre en oeuvre (pas de gestion du masking
sur des colonnes categorielles en texte) et valable quel que soit le nombre
d'interactions retenu par Optuna.

In [ ]:
import shap

preprocessor = best_estimator.named_steps["preprocessor"]
model = best_estimator.named_steps["model"]
feature_names = preprocessor.get_feature_names_out()

background = preprocessor.transform(X_train_val.sample(n=100, random_state=CONFIG["random_state"]))
explainer = shap.Explainer(model.predict, background, feature_names=feature_names)

sample_idx = 0
X_sample = preprocessor.transform(X_test.iloc[[sample_idx]])
shap_values = explainer(X_sample)

shap.plots.waterfall(shap_values[0])

Chaque barre correspond a une colonne encodee (`num__Rainfall_mm`,
`cat__Soil_Type_Clay`, ...) : une barre rouge pousse la prediction de
rendement vers le haut, une barre bleue vers le bas, et leur somme part de
la base `E[f(X)]` (prediction moyenne sur `background`) jusqu'a `f(x)`, la
prediction pour cet exemple precis. C'est ce niveau de detail, par variable
et par parcelle, qui justifie le choix d'EBM plutot qu'un modele boite
noire equivalent en performance (cf. section "Optimisation des
hyperparametres").